# 🔍 SOR PDF Diagnosis
### AI Estimation Agent — Nagaland PWD SOR 2021

This notebook diagnoses your SOR PDF and tells you:
- ✅ Is text extractable or is it a scanned image?
- ✅ Are tables detected?
- ✅ What do the pages look like?
- ✅ Exactly which extraction method to use next

**Run every cell top to bottom ↓**


## Step 1 — Install Libraries
*(Run once. Skip if already done.)*


In [9]:
!pip install pdfplumber pypdf pandas Pillow pdf2image -q
print("✅ Libraries ready")


✅ Libraries ready



[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## Step 2 — Imports


In [10]:
import os
import pdfplumber
from pypdf import PdfReader
import pandas as pd
from IPython.display import display, Image as IPImage
import warnings
warnings.filterwarnings("ignore")

print("Imports OK")


Imports OK


## Step 3 — Set Your SOR Filename
Make sure your SOR PDF is in the **same folder** as this notebook.


In [12]:
# ── Set your SOR PDF filename here ──────────────────────────
PDF_PATH = "NPWD-SOR-Volume-Isplit.pdf"    # ← change if your filename is different
# ─────────────────────────────────────────────────────────────

if os.path.exists(PDF_PATH):
    size_mb = os.path.getsize(PDF_PATH) / (1024 * 1024)
    print(f"File found : {PDF_PATH}")
    print(f"   Size       : {size_mb:.2f} MB")
else:
    print(f"File NOT found: {PDF_PATH}")
    print("   Make sure the PDF is in the same folder as this notebook.")


File found : NPWD-SOR-Volume-Isplit.pdf
   Size       : 1.66 MB


## Step 4 — Basic PDF Info


In [13]:
reader = PdfReader(PDF_PATH)

print("=" * 50)
print("  BASIC PDF INFORMATION")
print("=" * 50)
print(f"  Total pages  : {len(reader.pages)}")
print(f"  Encrypted    : {reader.is_encrypted}")

# Metadata
meta = reader.metadata
if meta:
    print(f"  Title        : {meta.title or '—'}")
    print(f"  Author       : {meta.author or '—'}")
    print(f"  Creator      : {meta.creator or '—'}")
    print(f"  Producer     : {meta.producer or '—'}")

TOTAL_PAGES = len(reader.pages)
print(f"\n  ✅ PDF has {TOTAL_PAGES} pages")


  BASIC PDF INFORMATION
  Total pages  : 288
  Encrypted    : False
  Title        : —
  Author       : —
  Creator      : —
  Producer     : iLovePDF

  ✅ PDF has 288 pages


In [15]:
INPUT_FILE  = "sor_ocr_raw.txt"   # your OCR output from Step 6
OUTPUT_CSV  = "sor_clean.csv"
OUTPUT_XLSX = "sor_clean.xlsx"

import os
if os.path.exists(INPUT_FILE):
    size = os.path.getsize(INPUT_FILE) / 1024
    print(f"File found: {INPUT_FILE} ({size:.1f} KB)")
else:
    print(f"ERROR: {INPUT_FILE} not found.")
    print("Run Step 6 of 0_OCR_SOR.ipynb first.")

File found: sor_ocr_raw.txt (796.6 KB)


## Step 5 — Text Extraction Check
This tells us if the PDF contains real selectable text or is a scanned image.
**This is the most important check.**


In [14]:
print("=" * 50)
print("  TEXT EXTRACTION CHECK")
print("=" * 50)

sample_pages = [0, 1, 2, 3, 4]   # check first 5 pages
results = []

with pdfplumber.open(PDF_PATH) as pdf:
    for i in sample_pages:
        if i >= len(pdf.pages):
            break
        page = pdf.pages[i]
        text = page.extract_text() or ""
        char_count = len(text.strip())
        results.append({"page": i+1, "chars": char_count, "sample": text[:120].replace("\n", " | ")})

df_text = pd.DataFrame(results)
display(df_text)

# Verdict
avg_chars = df_text["chars"].mean()
print()
if avg_chars > 300:
    print("✅ RESULT: TEXT-BASED PDF")
    print("   pdfplumber can extract text directly.")
    print("   → Proceed to 3_Extract.ipynb")
    PDF_TYPE = "text"
elif avg_chars > 50:
    print("⚠️  RESULT: PARTIAL TEXT")
    print("   Some pages have text, some may be scanned.")
    print("   → Try extraction first, use OCR for blank pages.")
    PDF_TYPE = "partial"
else:
    print("❌ RESULT: SCANNED / IMAGE PDF")
    print("   No extractable text found.")
    print("   → Need OCR step before extraction.")
    PDF_TYPE = "scanned"

print(f"\n   Average characters per page: {avg_chars:.0f}")


  TEXT EXTRACTION CHECK


,page,chars,sample
0,1,0,
1,2,0,
2,3,0,
3,4,2977,Code No. MF Description Unit Kohima Dimapur Pe...
4,5,3280,Code No. MF Description Unit Kohima Dimapur Pe...



✅ RESULT: TEXT-BASED PDF
   pdfplumber can extract text directly.
   → Proceed to 3_Extract.ipynb

   Average characters per page: 1251


## Step 6 — Table Detection
Checks if pdfplumber can automatically detect table structures.
This determines whether we use table extraction or text parsing.


In [16]:
print("=" * 50)
print("  TABLE DETECTION")
print("=" * 50)

table_results = []
sample_pages_tables = list(range(min(10, TOTAL_PAGES)))  # check first 10 pages

with pdfplumber.open(PDF_PATH) as pdf:
    for i in sample_pages_tables:
        page = pdf.pages[i]
        tables = page.extract_tables() or []
        
        table_info = {
            "page": i + 1,
            "tables_found": len(tables),
            "rows_in_first_table": len(tables[0]) if tables else 0,
            "cols_in_first_table": len(tables[0][0]) if tables and tables[0] else 0,
        }
        table_results.append(table_info)

df_tables = pd.DataFrame(table_results)
display(df_tables)

total_tables = df_tables["tables_found"].sum()
print()
if total_tables > 0:
    print(f"✅ Tables detected on {(df_tables['tables_found']>0).sum()} out of {len(sample_pages_tables)} pages")
    print("   → Use TABLE extraction mode in 3_Extract.ipynb")
    EXTRACTION_MODE = "table"
else:
    print("⚠️  No tables auto-detected")
    print("   SOR likely uses text layout (common in older PWD SORs)")
    print("   → Use TEXT PARSING mode in 3_Extract.ipynb")
    EXTRACTION_MODE = "text"
    
print(f"   Recommended extraction mode: {EXTRACTION_MODE.upper()}")


  TABLE DETECTION


,page,tables_found,rows_in_first_table,cols_in_first_table
0,1,0,0,0
1,2,0,0,0
2,3,0,0,0
3,4,1,17,16
4,5,1,15,16
5,6,1,27,16
6,7,1,20,16
7,8,1,17,16
8,9,1,19,16
9,10,2,1,16



✅ Tables detected on 7 out of 10 pages
   → Use TABLE extraction mode in 3_Extract.ipynb
   Recommended extraction mode: TABLE


## Step 7 — Preview Raw Text
Shows exactly what text is extracted from pages.
Look at this carefully — it tells us the structure of your SOR.


In [17]:
print("=" * 50)
print("  RAW TEXT PREVIEW — PAGES 1 to 5")
print("=" * 50)

with pdfplumber.open(PDF_PATH) as pdf:
    for i in range(min(5, TOTAL_PAGES)):
        page = pdf.pages[i]
        text = page.extract_text() or ""
        
        print(f"\n{'─'*45}")
        print(f"  PAGE {i+1}")
        print(f"{'─'*45}")
        
        if text.strip():
            # Show first 600 characters
            print(text[:600])
            if len(text) > 600:
                print(f"  ... ({len(text)} total characters on this page)")
        else:
            print("  [NO TEXT EXTRACTED — likely scanned image]")


  RAW TEXT PREVIEW — PAGES 1 to 5

─────────────────────────────────────────────
  PAGE 1
─────────────────────────────────────────────
  [NO TEXT EXTRACTED — likely scanned image]

─────────────────────────────────────────────
  PAGE 2
─────────────────────────────────────────────
  [NO TEXT EXTRACTED — likely scanned image]

─────────────────────────────────────────────
  PAGE 3
─────────────────────────────────────────────
  [NO TEXT EXTRACTED — likely scanned image]

─────────────────────────────────────────────
  PAGE 4
─────────────────────────────────────────────
Code No. MF Description Unit Kohima Dimapur Peren Wokha Phek Zunheboto Mokokchung Tuensang Mon Longleng Kiphire Noklak
2.0 EARTHWORK
Earth work in surface excavation not exceeding 30 cm in depth
butexceeding1.5minwidthaswellas10sqmonplanincluding
A 2.1
disposal of excavated earth upto 50 m and lift upto 1.5 m,
disposed soil to be levelled and neatly dressed:
A 2.1.1 All Kinds of Soil 100 Sqm 7465.70 7465.70 7465.70 7465

## Step 8 — Preview Table Content
If tables were detected, this shows the actual data inside them.
We're looking for: Item No, Description, Unit, Rate columns.


In [18]:
print("=" * 50)
print("  TABLE CONTENT PREVIEW")
print("=" * 50)

with pdfplumber.open(PDF_PATH) as pdf:
    tables_shown = 0
    for i in range(min(TOTAL_PAGES, 15)):  # scan first 15 pages
        page = pdf.pages[i]
        tables = page.extract_tables() or []
        
        for t_idx, table in enumerate(tables):
            if tables_shown >= 3:   # show max 3 tables
                break
            if len(table) < 2:
                continue
                
            print(f"\n  Page {i+1} — Table {t_idx+1} (first 8 rows):")
            print(f"  Rows: {len(table)}  |  Columns: {len(table[0]) if table else 0}")
            print()
            
            df = pd.DataFrame(table)
            display(df.head(8))
            tables_shown += 1
        
        if tables_shown >= 3:
            break

if tables_shown == 0:
    print("  No tables found in first 15 pages.")
    print("  SOR uses text layout — check Step 7 output above.")


  TABLE CONTENT PREVIEW

  Page 4 — Table 1 (first 8 rows):
  Rows: 17  |  Columns: 16



,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15
0,Code No.,MF,Description,Unit,Kohima,Dimapur,Peren,Wokha,Phek,Zunheboto,Mokokchung,Tuensang,Mon,Longleng,Kiphire,Noklak
1,NaN,NaN,2.0 EARTHWORK,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,A 2.1,,Earth work in surface excavation not exceeding...,,,,,,,,,,,,,
3,A 2.1.1,,All Kinds of Soil,100 Sqm,7465.70,7465.70,7465.70,7465.70,7465.70,7465.70,7465.70,7465.70,7465.70,7465.70,7465.70,7465.70
4,A 2.2,,"Earth work in rough excavation, banking excava...",,,,,,,,,,,,,
5,A 2.2.1,,All Kinds of soil,Cum,604.90,604.90,604.90,604.90,604.90,604.90,604.90,604.90,604.90,604.90,6.20,6.20
6,A 2.3,,Banking excavated earth in layers not exceedin...,,,,,,,,,,,,,
7,A 2.3.1,,All kinds of soil,Cum,382.70,382.60,382.70,382.80,382.90,382.80,382.70,382.90,382.70,382.80,383.00,382.90



  Page 5 — Table 1 (first 8 rows):
  Rows: 15  |  Columns: 16



,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15
0,Code No.,MF,Description,Unit,Kohima,Dimapur,Peren,Wokha,Phek,Zunheboto,Mokokchung,Tuensang,Mon,Longleng,Kiphire,Noklak
1,A 2.8,,Earth work in excavation by mechanical means (...,,,,,,,,,,,,,
2,A 2.8.1,,All Kinds of soil,Cum,278.20,278.20,278.20,278.20,278.20,278.20,278.20,278.20,278.20,278.20,278.20,278.20
3,A 2.9,,Earth work in excavation by mechanical means (...,,,,,,,,,,,,,
4,A 2.9.1,,Ordinary Rock,Cum,475.50,475.20,475.50,475.80,476.40,476.20,475.60,476.40,475.70,476.00,476.80,476.60
5,A 2.9.2,,Hard Rock ( Requiring Blasting),Cum,835.80,831.50,835.80,840.10,848.00,846.20,837.00,848.70,837.60,843.10,854.80,851.10
6,A 2.9.3,,Hard Rock (Blasting prohibited),Cum,1101.30,1101.00,1101.30,1101.70,1102.40,1102.30,1101.50,1102.50,1101.50,1102.00,1103.10,1102.70
7,A 2.10,,Excavating trenches of required width for pipe...,,,,,,,,,,,,,



  Page 6 — Table 1 (first 8 rows):
  Rows: 27  |  Columns: 16



,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15
0,Code No.,MF,Description,Unit,Kohima,Dimapur,Peren,Wokha,Phek,Zunheboto,Mokokchung,Tuensang,Mon,Longleng,Kiphire,Noklak
1,A 2.13.1,,Ordinary Rock,,,,,,,,,,,,,
2,A 2.13.1.1,,"Pipes, cables etc. not exceeding 80 mm dia.",Metre,315.10,315.00,315.10,315.30,315.60,315.50,315.20,315.60,315.20,315.40,315.80,315.70
3,A 2.13.1.2,,"Pipes, cables etc. exceeding 80 mm dia but not...",Metre,780.30,780.00,780.30,780.70,781.40,781.20,780.40,781.40,780.60,780.90,781.90,781.60
4,A 2.13.1.3,,"Pipes, cables exceeding 300 mm dia but not exc...",Metre,897.90,897.50,897.90,898.40,899.20,898.90,898.10,899.20,898.20,898.60,899.70,899.40
5,A 2.13.2,,Hard rock (requiring blasting),Metre,,,,,,,,,,,,
6,A 2.13.2.1,,"Pipes, cables etc. not exceeding 80 mm dia",Metre,487.10,485.00,487.10,489.10,492.80,492.00,487.60,493.10,487.90,490.50,496.00,494.30
7,A 2.13.2.2,,"Pipes, cables etc. exceeding 80 mm dia but not...",Metre,1206.00,1201.00,1206.00,1211.10,1220.30,1218.20,1207.40,1221.10,1208.10,1214.60,1228.30,1223.90


## Step 9 — Column Pattern Check
Searches for typical SOR patterns in the text:
item numbers, units (cum, sqm, etc.), and rates (numbers).


In [19]:
import re

print("=" * 50)
print("  SOR PATTERN CHECK")
print("=" * 50)

# Patterns we expect in a SOR
patterns = {
    "Item numbers (1.01, 2.3, etc.)":  r"\b\d+\.\d+\b",
    "Units (cum/sqm/rmt etc.)":         r"\b(cum|sqm|sqft|rmt|kg|nos|lump|litre|MT|RM|Rmt)\b",
    "Rates (numbers with decimals)":    r"\b\d{2,6}\.\d{2}\b",
    "Rs. symbol":                       r"(Rs\.?|₹|INR)",
}

with pdfplumber.open(PDF_PATH) as pdf:
    # Sample from pages 2-8 (skip cover page)
    sample_text = ""
    for i in range(1, min(8, TOTAL_PAGES)):
        text = pdf.pages[i].extract_text() or ""
        sample_text += text + "\n"

print(f"  Text sampled from pages 2-8: {len(sample_text)} characters\n")

for pattern_name, pattern in patterns.items():
    matches = re.findall(pattern, sample_text, re.IGNORECASE)
    unique = list(set(matches))[:10]
    status = "✅" if matches else "❌"
    print(f"  {status} {pattern_name}")
    print(f"     Found {len(matches)} matches — examples: {unique[:5]}")
    print()


  SOR PATTERN CHECK
  Text sampled from pages 2-8: 16315 characters

  ✅ Item numbers (1.01, 2.3, etc.)
     Found 877 matches — examples: ['1389.40', '145.60', '735.60', '2.1', '77.80']

  ✅ Units (cum/sqm/rmt etc.)
     Found 43 matches — examples: ['Sqm', 'Cum', 'cum']

  ✅ Rates (numbers with decimals)
     Found 718 matches — examples: ['1389.40', '145.60', '735.60', '77.80', '780.40']

  ✅ Rs. symbol
     Found 13 matches — examples: ['rs', 'inr']



## Step 10 — Visual Page Preview *(Optional)*
Renders a page as an image so you can see exactly what it looks like.
Useful if text extraction seems off.


In [21]:
# ── Set which page to preview (1 = first page) ──────────────
PAGE_TO_PREVIEW = 3     # ← change to any page number
# ─────────────────────────────────────────────────────────────
POPPLER_PATH = r"C:\poppler\Library\bin"
try:
    from pdf2image import convert_from_path
    pages = convert_from_path(PDF_PATH, dpi=150,
                              first_page=PAGE_TO_PREVIEW,
                              last_page=PAGE_TO_PREVIEW)
    img = pages[0]
    # Resize to reasonable display size
    w, h = img.size
    scale = min(900/w, 1100/h)
    img_resized = img.resize((int(w*scale), int(h*scale)))
    img_resized.save("/tmp/sor_preview.jpg", "JPEG")
    print(f"  Showing page {PAGE_TO_PREVIEW}:")
    display(IPImage("/tmp/sor_preview.jpg"))

except ImportError:
    print("  pdf2image not available.")
    print("  Install: pip install pdf2image")
    print("  Also needs poppler: https://github.com/oschwartz10612/poppler-windows/releases")
except Exception as e:
    print(f"  Could not render page: {e}")


  Could not render page: Unable to get page count. Is poppler installed and in PATH?


## Step 11 — Final Verdict & Next Steps


In [23]:
print("=" * 55)
print("  DIAGNOSIS COMPLETE — SUMMARY")
print("=" * 55)

print(f"""
  PDF File       : {PDF_PATH}
  Total Pages    : {TOTAL_PAGES}
  PDF Type       : {PDF_TYPE.upper()}
  Extraction Mode: {EXTRACTION_MODE.upper()}
""")

print("─" * 55)

if PDF_TYPE == "text" and EXTRACTION_MODE == "table":
    print("""
  [OK] BEST CASE — Text PDF with Tables

  Your SOR is directly readable.
  Go to: 3_Extract.ipynb
  Set  : MODE = "table"
""")

elif PDF_TYPE == "text" and EXTRACTION_MODE == "text":
    print("""
  [OK] GOOD — Text PDF, Text Layout

  Your SOR text is readable but uses text layout (no grid tables).
  Go to: 3_Extract.ipynb
  Set  : MODE = "text"

  You may need to adjust the regex pattern in Step 2
  based on what you saw in the raw text preview above.
""")

elif PDF_TYPE == "partial":
    print("""
  [WARN] MIXED — Some pages text, some scanned

  Try extraction first. Pages that come out blank need OCR.
  Go to: 3_Extract.ipynb
  Set  : MODE = "auto"
""")

elif PDF_TYPE == "scanned":
    print("""
  [!!] SCANNED PDF — Needs OCR

  The SOR was scanned as images. We need OCR first.
  Options:
  1. Use Adobe Acrobat → Tools → Enhance Scans → OCR
     (saves as searchable PDF → re-run this diagnosis)

  2. Free online OCR:
     https://www.ilovepdf.com/ocr-pdf
     (upload, download searchable PDF, rename to NPWD_SOR.pdf)

  3. Or tell me and I'll write an OCR script using
     pytesseract (needs Tesseract installed).
""")

print("─" * 55)
print("  Share this output if you need help — especially")
print("  the text preview from Step 7 and table preview")
print("  from Step 8.")
print("─" * 55)


  DIAGNOSIS COMPLETE — SUMMARY

  PDF File       : NPWD-SOR-Volume-Isplit.pdf
  Total Pages    : 288
  PDF Type       : TEXT
  Extraction Mode: TABLE

───────────────────────────────────────────────────────

  [OK] BEST CASE — Text PDF with Tables

  Your SOR is directly readable.
  Go to: 3_Extract.ipynb
  Set  : MODE = "table"

───────────────────────────────────────────────────────
  Share this output if you need help — especially
  the text preview from Step 7 and table preview
  from Step 8.
───────────────────────────────────────────────────────
